# Formulaic prose → natural prose: seq2seq PoC

This notebook fine-tunes `google/flan-t5-small` on paired rewrites. The bundled data is a **pipeline smoke test only**. Upload real, human-reviewed pairs before interpreting model quality.

In [ ]:
!pip -q install 'transformers>=4.48,<5' 'datasets>=3.2,<4' 'accelerate>=1.2,<2' 'evaluate>=0.4,<1' 'sentence-transformers>=3.3,<4' 'sacrebleu>=2.4,<3'

In [ ]:
import os, re, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments)

SEED = 42
MODEL_NAME = 'google/flan-t5-small'
OUTPUT_DIR = Path('artifacts/flan-t5-small-naturalizer')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## Load paired data
Upload `pairs.csv` with `source`, `target`, and `group_id`. You can upload the repository's `data/example_pairs.csv` for a smoke test. Rows sharing an origin, document, prompt, or topic should share a `group_id`.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert uploaded, 'Upload a CSV file to continue.'
data_path = next(iter(uploaded))

df = pd.read_csv(data_path).dropna(subset=['source', 'target', 'group_id'])
required = {'source', 'target', 'group_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
df = df.drop_duplicates(subset=['source', 'target']).reset_index(drop=True)
assert len(df) >= 10 and df.group_id.nunique() >= 5, 'Need at least 10 rows across 5 groups for a smoke test.'
print(f'{len(df):,} pairs across {df.group_id.nunique():,} groups')
df.sample(min(3, len(df)), random_state=SEED)

In [ ]:
# 80/10/10-ish split by group. Small datasets may have slightly different proportions.
outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, held_idx = next(outer.split(df, groups=df.group_id))
train_df, held_df = df.iloc[train_idx], df.iloc[held_idx]
inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel, test_rel = next(inner.split(held_df, groups=held_df.group_id))
val_df, test_df = held_df.iloc[val_rel], held_df.iloc[test_rel]
splits = {'train': train_df, 'validation': val_df, 'test': test_df}
assert not (set(train_df.group_id) & set(val_df.group_id) | set(train_df.group_id) & set(test_df.group_id) | set(val_df.group_id) & set(test_df.group_id))
print({name: (len(part), part.group_id.nunique()) for name, part in splits.items()})

## Tokenize and fine-tune

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
prefix = 'Rewrite this in natural, direct prose without changing its meaning: '
raw = DatasetDict({k: Dataset.from_pandas(v[['source', 'target']], preserve_index=False) for k, v in splits.items()})

def tokenize(batch):
    inputs = tokenizer([prefix + x for x in batch['source']], max_length=384, truncation=True)
    labels = tokenizer(text_target=batch['target'], max_length=384, truncation=True)
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized = raw.map(tokenize, batched=True, remove_columns=raw['train'].column_names)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR), learning_rate=2e-4,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    gradient_accumulation_steps=2, num_train_epochs=5, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    predict_with_generate=True, generation_max_length=384,
    fp16=torch.cuda.is_available(), report_to='none', seed=SEED,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'], processing_class=tokenizer, data_collator=collator)
trainer.train()
trainer.save_model(str(OUTPUT_DIR)); tokenizer.save_pretrained(str(OUTPUT_DIR))

## Held-out evaluation
SARI measures editing against the reference. Embedding similarity and entity/number preservation are safety indicators, not proof of equivalent meaning. The final decision requires blind human review.

In [ ]:
pred = trainer.predict(tokenized['test'])
pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
outputs = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
results = test_df[['group_id', 'source', 'target']].reset_index(drop=True).copy()
results['prediction'] = outputs
results

In [ ]:
import evaluate
from sentence_transformers import SentenceTransformer

sari = evaluate.load('sari')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def semantic_similarity(sources, candidates):
    a = embedder.encode(list(sources), normalize_embeddings=True)
    b = embedder.encode(list(candidates), normalize_embeddings=True)
    return np.sum(a * b, axis=1)

def protected_tokens(text):
    # Review this heuristic manually: capitalization is an imperfect proxy for names.
    numbers = re.findall(r'(?<!\w)[+-]?(?:\d[\d,.]*%?)(?!\w)', text)
    names = re.findall(r'(?<![.!?]\s)\b[A-Z][A-Za-z0-9_-]+\b', text)
    return set(numbers + names)

def preservation_rate(sources, candidates):
    scores = []
    for src, cand in zip(sources, candidates):
        tokens = protected_tokens(src)
        scores.append(1.0 if not tokens else len(tokens & protected_tokens(cand)) / len(tokens))
    return np.array(scores)

def score(label, candidates):
    return {
        'system': label,
        'SARI': sari.compute(sources=results.source.tolist(), predictions=list(candidates), references=[[x] for x in results.target])['sari'],
        'source_similarity': semantic_similarity(results.source, candidates).mean(),
        'protected_token_recall': preservation_rate(results.source, candidates).mean(),
    }

metrics = pd.DataFrame([score('identity baseline', results.source), score('fine-tuned model', results.prediction)])
metrics

In [ ]:
# Export predictions plus a blinded A/B sheet for human review.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results['source_similarity'] = semantic_similarity(results.source, results.prediction)
results['protected_token_recall'] = preservation_rate(results.source, results.prediction)
results.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
rng = np.random.default_rng(SEED)
model_is_a = rng.random(len(results)) < 0.5
review = pd.DataFrame({
    'item_id': np.arange(len(results)), 'source': results.source,
    'option_a': np.where(model_is_a, results.prediction, results.source),
    'option_b': np.where(model_is_a, results.source, results.prediction),
    'naturalness_winner_A_B_TIE': '', 'a_preserves_meaning_Y_N': '',
    'b_preserves_meaning_Y_N': '', 'notes': ''})
key = pd.DataFrame({'item_id': review.item_id, 'model_option': np.where(model_is_a, 'A', 'B')})
review.to_csv(OUTPUT_DIR / 'blind_review.csv', index=False)
key.to_csv(OUTPUT_DIR / 'blind_review_key.csv', index=False)
shutil.make_archive('naturalizer_artifacts', 'zip', 'artifacts')
files.download('naturalizer_artifacts.zip')